## 0. Setup

Sesuaikan `DATA_DIR` di bawah ini ke folder dataset kompetisi kamu
(folder yang berisi `train.csv`, `test.csv`, `data_pendukung/`, dst).


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import lightgbm as lgb
from sklearn.metrics import mean_squared_error

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', None)

# ---- CONFIG: sesuaikan path ini ----
DATA_DIR = Path('.')            # folder dataset (berisi train.csv, test.csv, data_pendukung/)
OUTPUT_DIR = Path('./output')   # folder untuk simpan file perantara & hasil
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VAL_CUTOFF = '2025-05-19'   # tanggal potong untuk validasi time-based
NUM_BOOST_ROUND = 3000
EARLY_STOPPING = 100
FENCE_K = 3.0               # Tukey-fence multiplier untuk clipping
BBOX_BUFFER = 0.3           # buffer (derajat) untuk filter shapefile HydroRIVERS

## 1. EDA singkat

Struktur data: 30 pos pemantauan, target TMA 3x/hari (06:00/12:00/18:00),
periode Jan 2023 – Mei 2026 (train sampai Sep 2025, test 8 bulan ke depan).


In [2]:
train_eda = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['datetime'])
test_raw_eda = pd.read_csv(DATA_DIR / 'test.csv')

print("Train shape:", train_eda.shape, "| N pos:", train_eda['nama_pos'].nunique())
print("Date range:", train_eda['datetime'].min(), "-", train_eda['datetime'].max())
print()
print("Statistik TMA per pos (5 contoh):")
print(train_eda.groupby('nama_pos')['tma_mdpl'].agg(['min','max','mean','std']).head())

Train shape: (84396, 3) | N pos: 30
Date range: 2023-01-01 06:00:00 - 2025-09-18 18:00:00

Statistik TMA per pos (5 contoh):
                               min         max        mean       std
nama_pos                                                            
Arjowinangun - Pacitan    0.347639    4.850000    1.116478  0.494220
Babat                     4.410000    9.660000    6.305936  0.770386
Badegan                 121.902101  123.960000  122.402005  0.309940
Bengkelolor               7.588552   13.522786    9.813831  1.376864
Boboh Kali Lamong         2.100000    7.600000    4.107014  1.289734


## 2. Fitur eksogen (cuaca/tanah/iklim)

Agregasi `data_lingkungan.csv` (per jam) menjadi fitur rolling di jendela
6h/24h/72h/168h, disejajarkan dengan waktu target TMA (06:00/12:00/18:00).


In [3]:
import argparse
from pathlib import Path
import pandas as pd
import numpy as np

data_dir = DATA_DIR
out_dir = OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

env_path = data_dir / 'data_pendukung' / 'data_lingkungan.csv'
train_path = data_dir / 'train.csv'
test_path = data_dir / 'test.csv'

print(f"Loading {env_path} ...")
env = pd.read_csv(env_path, parse_dates=['datetime'])
env = env.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

print(f"Loading {train_path} and {test_path} ...")
train = pd.read_csv(train_path, parse_dates=['datetime'])
test_raw = pd.read_csv(test_path)
test = pd.DataFrame()
test['datetime'] = pd.to_datetime(test_raw['id'].str.split(' - ', n=1).str[0])
test['nama_pos'] = test_raw['id'].str.split(' - ', n=1).str[1]
test['id'] = test_raw['id']

targets = pd.concat([
    train[['datetime', 'nama_pos']].assign(split='train'),
    test[['datetime', 'nama_pos']].assign(split='test')
], ignore_index=True)
print("Total target rows:", len(targets))

static_cols = ['built_surface_m2', 'landcover_class']
rain_cols = ['rainfall_mm', 'rainfall_openmeteo_mm']
mean_cols = ['humidity_pct', 'wind_direction_deg', 'dew_point_c', 'cloud_cover_pct',
             'temperature_c', 'wind_speed_kmh', 'solar_radiation_mj_m2',
             'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm',
             'soil_moisture_100_255cm', 'surface_pressure_hpa', 'pressure_msl_hpa']
climate_cols = ['rmm1', 'rmm2', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'nino_34']
already_agg_cols = ['rainfall_max_24h_mm']

windows_hours = [6, 24, 72, 168]

results = []
pos_list = env['nama_pos'].unique()

for pos in pos_list:
    env_p = env[env['nama_pos'] == pos].set_index('datetime').sort_index()
    tgt_p = targets[targets['nama_pos'] == pos].sort_values('datetime')
    if len(tgt_p) == 0:
        continue

    full_idx = pd.date_range(env_p.index.min(), env_p.index.max(), freq='h')
    env_p = env_p.reindex(full_idx)
    env_p[mean_cols + rain_cols + climate_cols + already_agg_cols] = \
        env_p[mean_cols + rain_cols + climate_cols + already_agg_cols].ffill(limit=6)
    env_p[static_cols] = env_p[static_cols].ffill().bfill()

    feat = pd.DataFrame(index=tgt_p['datetime'].values)

    inst_cols = mean_cols + climate_cols + already_agg_cols + static_cols
    inst = env_p[inst_cols].reindex(feat.index, method='ffill')
    inst = inst.add_suffix('_now')
    feat = feat.join(inst)

    for w in windows_hours:
        for c in rain_cols:
            s = env_p[c].rolling(f'{w}h', min_periods=1).sum()
            feat[f'{c}_sum_{w}h'] = s.reindex(feat.index, method='ffill').values
        for c in mean_cols:
            s = env_p[c].rolling(f'{w}h', min_periods=1).mean()
            feat[f'{c}_mean_{w}h'] = s.reindex(feat.index, method='ffill').values

    feat['nama_pos'] = pos
    feat['datetime'] = feat.index
    results.append(feat.reset_index(drop=True))
    print(f"  done: {pos} ({len(feat)} rows)")

exog_feat = pd.concat(results, ignore_index=True)
print("Final exog feature shape:", exog_feat.shape)

out_path = out_dir / 'exog_features.parquet'
exog_feat.to_parquet(out_path, index=False)
print(f"Saved to {out_path}")

Loading data_pendukung/data_lingkungan.csv ...


Loading train.csv and test.csv ...
Total target rows: 106176


  done: Arjowinangun - Pacitan (3629 rows)
  done: Babat (3564 rows)


  done: Badegan (3627 rows)
  done: Bengkelolor (3627 rows)


  done: Boboh Kali Lamong (3628 rows)
  done: Bojonegoro - Kali Kethek (3414 rows)


...

  done: Napel (3627 rows)
  done: Ngadipiro (3627 rows)


  done: Ngrembang (3627 rows)
  done: Peren (3628 rows)


  done: Sekayu (3627 rows)
  done: Serenan (3627 rows)


  done: Sumberrejo (3604 rows)
  done: Wonogiri Dam (3627 rows)
Final exog feature shape: (106176, 84)


Saved to output/exog_features.parquet


## 3. Fitur hulu-hilir (topologi sungai dari HydroRIVERS)

Snap tiap 30 pos ke segmen sungai HydroRIVERS terdekat, lalu urutkan dari
hilir ke hulu memakai `DIST_DN_KM` (jarak ke muara). Hasilnya: mapping
"pos ini punya pos hulu apa" yang dipakai sebagai fitur di langkah berikutnya.

> Catatan: 26 dari 30 pos ternyata berada di mainstem Bengawan Solo yang sama;
> sisanya (`Boboh Kali Lamong`, `Bengkelolor`, `Gunungsari`,
> `Arjowinangun - Pacitan`, `Lorog`) berada di anak sungai/DAS terpisah.

Butuh `geopandas` dan `shapely` (`pip install geopandas shapely`).


In [4]:
import warnings
import geopandas as gpd
from shapely.geometry import box
warnings.filterwarnings('ignore', category=UserWarning)

import argparse
from pathlib import Path
import warnings
import geopandas as gpd
import pandas as pd
from shapely.geometry import box

warnings.filterwarnings('ignore', category=UserWarning)

data_dir = DATA_DIR
out_dir = OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

shp_path = data_dir / 'data_pendukung' / 'HydroRIVERS_v10_au_shp' / 'HydroRIVERS_v10_au.shp'
koord_path = data_dir / 'data_pendukung' / 'koordinat_pos.csv'

print(f"Loading {shp_path} ...")
gdf = gpd.read_file(shp_path)
koord = pd.read_csv(koord_path)

b = BBOX_BUFFER
bbox = box(koord['longitude'].min() - b, koord['latitude'].min() - b,
           koord['longitude'].max() + b, koord['latitude'].max() + b)
sub = gdf[gdf.intersects(bbox)].copy()
print(f"Filtered to {len(sub)} river segments near pos bounding box")

pts = gpd.GeoDataFrame(koord, geometry=gpd.points_from_xy(koord['longitude'], koord['latitude']), crs='EPSG:4326')
nearest = gpd.sjoin_nearest(
    pts, sub[['HYRIV_ID', 'NEXT_DOWN', 'MAIN_RIV', 'DIST_DN_KM', 'ORD_STRA', 'geometry']],
    how='left', distance_col='dist_deg'
)
nearest = nearest.drop_duplicates(subset='nama_pos').reset_index(drop=True)

upstream_map, dist_to_upstream = {}, {}
for main_riv, g in nearest.groupby('MAIN_RIV'):
    g = g.sort_values('DIST_DN_KM').reset_index(drop=True)
    for i in range(len(g)):
        if i == 0:
            upstream_map[g.loc[i, 'nama_pos']] = None
            dist_to_upstream[g.loc[i, 'nama_pos']] = None
        else:
            upstream_map[g.loc[i, 'nama_pos']] = g.loc[i - 1, 'nama_pos']
            dist_to_upstream[g.loc[i, 'nama_pos']] = g.loc[i, 'DIST_DN_KM'] - g.loc[i - 1, 'DIST_DN_KM']

result = nearest[['nama_pos', 'MAIN_RIV', 'DIST_DN_KM']].copy()
result['upstream_pos'] = result['nama_pos'].map(upstream_map)
result['dist_to_upstream_km'] = result['nama_pos'].map(dist_to_upstream)
result = result.sort_values(['MAIN_RIV', 'DIST_DN_KM'])

print(result.to_string())

out_path = data_dir / 'data_pendukung' / 'upstream_mapping.csv'
result.to_csv(out_path, index=False)
print(f"\nSaved {out_path}")
print(f"{result['upstream_pos'].notna().sum()} / {len(result)} pos have an identified upstream neighbor")

Loading data_pendukung/HydroRIVERS_v10_au_shp/HydroRIVERS_v10_au.shp ...


Filtered to 5361 river segments near pos bounding box
                     nama_pos  MAIN_RIV  DIST_DN_KM               upstream_pos  dist_to_upstream_km
3                Karanggeneng  50387677        35.1                        NaN                  NaN
27          Floodway Bridge C  50387677        61.9               Karanggeneng                 26.8
20                      Babat  50387677        83.2          Floodway Bridge C                 21.3
26                 Sumberrejo  50387677       107.1                      Babat                 23.9
4    Bojonegoro - Kali Kethek  50387677       134.7                 Sumberrejo                 27.6
18                   Brangkal  50387677       140.0   Bojonegoro - Kali Kethek                  5.3
2                        Cepu  50387677       194.2                   Brangkal                 54.2
15               Karangnongko  50387677       212.9                       Cepu                 18.7
21                      Napel  50387677       

## 4. Grid waktu penuh + lag features + outlier capping

Bangun grid waktu 6-jam-an lengkap per pos (train + test), capping outlier
ringan pada target (persentil 0.1%/99.9%), dan gabungkan mapping hulu-hilir.


In [5]:
import argparse
from pathlib import Path
import pandas as pd
import numpy as np

data_dir = DATA_DIR
out_dir = OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(data_dir / 'train.csv', parse_dates=['datetime'])
test_raw = pd.read_csv(data_dir / 'test.csv')
test = pd.DataFrame()
test['datetime'] = pd.to_datetime(test_raw['id'].str.split(' - ', n=1).str[0])
test['nama_pos'] = test_raw['id'].str.split(' - ', n=1).str[1]
test['id'] = test_raw['id']
test['tma_mdpl'] = np.nan

koord = pd.read_csv(data_dir / 'data_pendukung' / 'koordinat_pos.csv')

# upstream_mapping.csv: derived from HydroRIVERS shapefile (see 06_build_upstream_mapping.py).
# Maps each pos to its nearest upstream neighbor along the same river mainstem.
upstream_path = data_dir / 'data_pendukung' / 'upstream_mapping.csv'
if upstream_path.exists():
    upmap = pd.read_csv(upstream_path)[['nama_pos', 'upstream_pos', 'dist_to_upstream_km']]
    koord = koord.merge(upmap, on='nama_pos', how='left')
    print(f"Merged upstream_pos mapping ({upmap['upstream_pos'].notna().sum()} pos have an upstream neighbor)")
else:
    koord['upstream_pos'] = None
    koord['dist_to_upstream_km'] = None
    print("WARNING: upstream_mapping.csv not found, skipping upstream features")

full = pd.concat([
    train[['datetime', 'nama_pos', 'tma_mdpl']].assign(split='train'),
    test[['datetime', 'nama_pos', 'tma_mdpl']].assign(split='test')
], ignore_index=True).sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

# ---- Outlier capping on TRAIN target only (per pos, percentile-based) ----
train_mask = full['split'] == 'train'
lo = full.loc[train_mask].groupby('nama_pos')['tma_mdpl'].transform(lambda s: s.quantile(0.001))
hi = full.loc[train_mask].groupby('nama_pos')['tma_mdpl'].transform(lambda s: s.quantile(0.999))
full.loc[train_mask, 'tma_mdpl'] = full.loc[train_mask, 'tma_mdpl'].clip(lower=lo, upper=hi)

# ---- Build a complete 6-hourly grid per pos (train may have missing steps) ----
all_rows = []
for pos, g in full.groupby('nama_pos'):
    g = g.set_index('datetime').sort_index()
    full_idx = pd.date_range(g.index.min(), g.index.max(), freq='6h')
    g2 = g.reindex(full_idx)
    g2['nama_pos'] = pos
    g2['split'] = g2['split'].fillna('gap')
    g2['datetime'] = g2.index
    all_rows.append(g2.reset_index(drop=True))

full = pd.concat(all_rows, ignore_index=True)
full = full.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

print("Full grid shape (incl gaps):", full.shape)
print(full['split'].value_counts())

full = full.merge(koord, on='nama_pos', how='left')

out_path = out_dir / 'full_grid.parquet'
full.to_parquet(out_path, index=False)
print(f"Saved {out_path}")
print(full.head(10))

Merged upstream_pos mapping (26 pos have an upstream neighbor)
Full grid shape (incl gaps): (145688, 4)
split
train    84396
gap      39512
test     21780
Name: count, dtype: int64


Saved output/full_grid.parquet
                 nama_pos  tma_mdpl  split            datetime  latitude   longitude upstream_pos  dist_to_upstream_km
0  Arjowinangun - Pacitan      1.30  train 2023-01-01 06:00:00 -8.197561  111.114314          NaN                  NaN
1  Arjowinangun - Pacitan      1.20  train 2023-01-01 12:00:00 -8.197561  111.114314          NaN                  NaN
2  Arjowinangun - Pacitan      1.50  train 2023-01-01 18:00:00 -8.197561  111.114314          NaN                  NaN
3  Arjowinangun - Pacitan       NaN    gap 2023-01-02 00:00:00 -8.197561  111.114314          NaN                  NaN
4  Arjowinangun - Pacitan      1.45  train 2023-01-02 06:00:00 -8.197561  111.114314          NaN                  NaN
5  Arjowinangun - Pacitan      1.25  train 2023-01-02 12:00:00 -8.197561  111.114314          NaN                  NaN
6  Arjowinangun - Pacitan      1.60  train 2023-01-02 18:00:00 -8.197561  111.114314          NaN                  NaN
7  Arjowinangun -

## 5. Training LightGBM + validasi (one-step)

**Catatan penting:** validasi di sini bersifat *one-step* (pakai lag dari data
historis asli), BUKAN simulasi recursive forecasting yang sesungguhnya
dipakai untuk memprediksi test 8 bulan ke depan. Angka RMSE di sini
cenderung terlalu optimis — lihat Bagian 6 untuk estimasi yang jujur.


In [6]:
import argparse
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error

pd.set_option('display.width', 160)

LAGS = [1, 2, 3, 4, 7, 28]


def add_lag_features(d):
    d = d.sort_values(['nama_pos', 'datetime']).copy()
    g = d.groupby('nama_pos')['tma_mdpl']
    for lag in LAGS:
        d[f'lag_{lag}'] = g.shift(lag)
    shifted = g.shift(1)
    d['roll_mean_4'] = shifted.groupby(d['nama_pos']).transform(lambda s: s.rolling(4, min_periods=1).mean())
    d['roll_mean_28'] = shifted.groupby(d['nama_pos']).transform(lambda s: s.rolling(28, min_periods=1).mean())
    d['roll_std_28'] = shifted.groupby(d['nama_pos']).transform(lambda s: s.rolling(28, min_periods=2).std())
    d['diff_1'] = d['lag_1'] - d['lag_2']

    # ---- Upstream features: pull the upstream pos's own lag_1/lag_2/diff_1 at the SAME
    # datetime. This is safe for recursive forecasting because lag_1/lag_2 of the upstream
    # pos only depend on timesteps strictly before `datetime`, so no within-step ordering
    # is required between pos. ----
    if 'upstream_pos' in d.columns:
        up_cols = ['nama_pos', 'datetime', 'lag_1', 'lag_2', 'lag_3', 'diff_1', 'roll_mean_4']
        up_data = d[up_cols].rename(columns={
            'nama_pos': 'upstream_pos',
            'lag_1': 'up_lag_1', 'lag_2': 'up_lag_2', 'lag_3': 'up_lag_3',
            'diff_1': 'up_diff_1', 'roll_mean_4': 'up_roll_mean_4'
        })
        d = d.merge(up_data, on=['upstream_pos', 'datetime'], how='left')
    return d

out_dir = OUTPUT_DIR

full = pd.read_parquet(out_dir / 'full_grid.parquet')
exog = pd.read_parquet(out_dir / 'exog_features.parquet')

full = full.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)
exog = exog.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

df = full.merge(exog, on=['nama_pos', 'datetime'], how='left')
print("Merged shape:", df.shape)

df['hour'] = df['datetime'].dt.hour
df['month'] = df['datetime'].dt.month
doy = df['datetime'].dt.dayofyear
df['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
df['nama_pos_cat'] = df['nama_pos'].astype('category')

df = add_lag_features(df)

feature_cols = (
    [f'lag_{l}' for l in LAGS] +
    ['roll_mean_4', 'roll_mean_28', 'roll_std_28', 'diff_1',
     'up_lag_1', 'up_lag_2', 'up_lag_3', 'up_diff_1', 'up_roll_mean_4', 'dist_to_upstream_km',
     'hour', 'month', 'doy_sin', 'doy_cos', 'latitude', 'longitude'] +
    [c for c in exog.columns if c not in ('nama_pos', 'datetime')]
)
cat_features = ['nama_pos_cat']
all_feats = feature_cols + cat_features
print("N features:", len(all_feats))

train_df = df[df['split'] == 'train'].copy()

val_cutoff = pd.Timestamp(VAL_CUTOFF)
tr = train_df[train_df['datetime'] < val_cutoff]
va = train_df[train_df['datetime'] >= val_cutoff]
print("Train rows:", len(tr), "Val rows:", len(va))

X_tr, y_tr = tr[all_feats], tr['tma_mdpl']
X_va, y_va = va[all_feats], va['tma_mdpl']

lgb_train = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_features)
lgb_val = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_features, reference=lgb_train)

params = dict(
    objective='regression', metric='rmse', learning_rate=0.03,
    num_leaves=63, min_data_in_leaf=30, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=1, verbose=-1,
)

model = lgb.train(
    params, lgb_train, num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[lgb_train, lgb_val], valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(EARLY_STOPPING), lgb.log_evaluation(100)]
)

pred_va = model.predict(X_va, num_iteration=model.best_iteration)
rmse = np.sqrt(mean_squared_error(y_va, pred_va))
print(f"\nValidation RMSE (global, all pos mixed): {rmse:.4f}")

va_res = va.copy()
va_res['pred'] = pred_va
per_pos_rmse = va_res.groupby('nama_pos').apply(
    lambda g: np.sqrt(mean_squared_error(g['tma_mdpl'], g['pred']))
).sort_values(ascending=False)
print("\nWorst 10 pos by RMSE:\n", per_pos_rmse.head(10))
print("\nBest 10 pos by RMSE:\n", per_pos_rmse.tail(10))

model.save_model(str(out_dir / 'lgb_model_val.txt'))

with open(out_dir / 'feature_cols.pkl', 'wb') as f:
    pickle.dump({'feature_cols': feature_cols, 'cat_features': cat_features,
                 'all_feats': all_feats, 'best_iteration': model.best_iteration}, f)

print(f"\nSaved model + feature list to {out_dir}")

Merged shape: (145688, 90)


N features: 105
Train rows: 73352 Val rows: 11044


Training until validation scores don't improve for 100 rounds


[100]	train's rmse: 2.29802	val's rmse: 2.238


[200]	train's rmse: 0.521322	val's rmse: 0.291501


[300]	train's rmse: 0.463954	val's rmse: 0.274178


[400]	train's rmse: 0.433635	val's rmse: 0.274373


Early stopping, best iteration is:
[328]	train's rmse: 0.454125	val's rmse: 0.273862



Validation RMSE (global, all pos mixed): 0.2739

Worst 10 pos by RMSE:
 nama_pos
Bojonegoro - Kali Kethek    0.485940
Kedungupit                  0.424667
Brangkal                    0.400811
Floodway Bridge C           0.400473
Napel                       0.367765
Ketonggo                    0.366368
Karangnongko                0.353752
Jurug                       0.322211
Bengkelolor                 0.303272
Jarum                       0.296698
dtype: float64

Best 10 pos by RMSE:
 nama_pos
Kali Pepe - PTPN         0.174343
Karanggeneng             0.170045
Sekayu                   0.168016
Gunungsari               0.159377
Serenan                  0.147588
Kali Pepe - Tugu Boto    0.135408
Colo Weir                0.127808
Lorog                    0.122769
Ngadipiro                0.096551
Ngrembang                0.082946
dtype: float64

Saved model + feature list to output


## 6. Validasi recursive yang jujur

Mensimulasikan proses recursive/autoregressive yang SAMA seperti yang dipakai
untuk forecast test (prediksi langkah t dipakai sebagai lag untuk t+1, dst),
tapi pada periode hold-out yang nilai aslinya kita tahu. Ini mengungkap
**drift** yang menumpuk (terutama untuk pos dengan variansi historis sangat
kecil, seperti `Lorog`), dan menguji apakah **Tukey-fence clipping** membantu.

Hasil eksperimen kami:

| Metode | RMSE recursive |
|---|---|
| Tanpa clipping | 2.08 |
| Dengan clipping | 1.54 |
| Dengan clipping + fitur hulu-hilir | **1.47** |


In [7]:
import argparse
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

LAGS = [1, 2, 3, 4, 7, 28]


def add_lag_features(d):
    d = d.sort_values(['nama_pos', 'datetime']).copy()
    g = d.groupby('nama_pos')['tma_mdpl']
    for lag in LAGS:
        d[f'lag_{lag}'] = g.shift(lag)
    shifted = g.shift(1)
    d['roll_mean_4'] = shifted.groupby(d['nama_pos']).transform(lambda s: s.rolling(4, min_periods=1).mean())
    d['roll_mean_28'] = shifted.groupby(d['nama_pos']).transform(lambda s: s.rolling(28, min_periods=1).mean())
    d['roll_std_28'] = shifted.groupby(d['nama_pos']).transform(lambda s: s.rolling(28, min_periods=2).std())
    d['diff_1'] = d['lag_1'] - d['lag_2']

    if 'upstream_pos' in d.columns:
        up_cols = ['nama_pos', 'datetime', 'lag_1', 'lag_2', 'lag_3', 'diff_1', 'roll_mean_4']
        up_data = d[up_cols].rename(columns={
            'nama_pos': 'upstream_pos',
            'lag_1': 'up_lag_1', 'lag_2': 'up_lag_2', 'lag_3': 'up_lag_3',
            'diff_1': 'up_diff_1', 'roll_mean_4': 'up_roll_mean_4'
        })
        d = d.merge(up_data, on=['upstream_pos', 'datetime'], how='left')
    return d


def get_lag_val(series, ts, steps_back):
    idx = series.index
    pos_loc = idx.searchsorted(ts)
    target_loc = pos_loc - steps_back
    if target_loc < 0:
        return np.nan
    return series.iloc[target_loc]


def recursive_forecast(model, work, exog_cols, all_feats, forecast_datetimes, pos_list,
                        clip_bounds=None):
    """clip_bounds: dict pos -> (lo, hi) or None for no clipping."""
    series_by_pos = {p: work.xs(p, level=0)['tma_mdpl'].sort_index().copy() for p in pos_list}
    # blank out the actual values at forecast_datetimes so we forecast them fresh
    for p in pos_list:
        s = series_by_pos[p]
        mask = s.index.isin(forecast_datetimes)
        s.loc[mask] = np.nan

    static_row_cache = {p: work.xs(p, level=0) for p in pos_list}

    results = []
    for ts in forecast_datetimes:
        rows_feat = []
        for p in pos_list:
            srow = static_row_cache[p]
            if ts not in srow.index:
                continue
            series = series_by_pos[p]
            lag_vals = {f'lag_{l}': get_lag_val(series, ts, l) for l in LAGS}

            idxloc = series.index.searchsorted(ts)
            past = series.iloc[max(0, idxloc - 28):idxloc]
            past4 = past.iloc[-4:] if len(past) >= 1 else past
            roll_mean_4 = past4.mean() if len(past4) > 0 else np.nan
            roll_mean_28 = past.mean() if len(past) > 0 else np.nan
            roll_std_28 = past.std() if len(past) > 1 else np.nan
            diff_1 = (lag_vals['lag_1'] - lag_vals['lag_2']
                      if pd.notna(lag_vals['lag_1']) and pd.notna(lag_vals['lag_2']) else np.nan)

            row = srow.loc[ts].copy()
            feat_row = {c: row[c] for c in exog_cols}
            feat_row.update(lag_vals)
            feat_row['roll_mean_4'] = roll_mean_4
            feat_row['roll_mean_28'] = roll_mean_28
            feat_row['roll_std_28'] = roll_std_28
            feat_row['diff_1'] = diff_1

            up_pos = row.get('upstream_pos', None)
            if pd.notna(up_pos) and up_pos in series_by_pos:
                up_series = series_by_pos[up_pos]
                up_lag1 = get_lag_val(up_series, ts, 1)
                up_lag2 = get_lag_val(up_series, ts, 2)
                up_lag3 = get_lag_val(up_series, ts, 3)
                up_idxloc = up_series.index.searchsorted(ts)
                up_past4 = up_series.iloc[max(0, up_idxloc - 1 - 4):up_idxloc - 1] if up_idxloc >= 1 else pd.Series(dtype=float)
                feat_row['up_lag_1'] = up_lag1
                feat_row['up_lag_2'] = up_lag2
                feat_row['up_lag_3'] = up_lag3
                feat_row['up_diff_1'] = (up_lag1 - up_lag2) if pd.notna(up_lag1) and pd.notna(up_lag2) else np.nan
                feat_row['up_roll_mean_4'] = up_past4.mean() if len(up_past4) > 0 else np.nan
            else:
                feat_row['up_lag_1'] = np.nan
                feat_row['up_lag_2'] = np.nan
                feat_row['up_lag_3'] = np.nan
                feat_row['up_diff_1'] = np.nan
                feat_row['up_roll_mean_4'] = np.nan
            feat_row['dist_to_upstream_km'] = row.get('dist_to_upstream_km', np.nan)

            feat_row['hour'] = ts.hour
            feat_row['month'] = ts.month
            doy = ts.dayofyear
            feat_row['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
            feat_row['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
            feat_row['latitude'] = row['latitude']
            feat_row['longitude'] = row['longitude']
            feat_row['nama_pos_cat'] = p
            feat_row['nama_pos'] = p
            feat_row['datetime'] = ts
            rows_feat.append(feat_row)

        batch = pd.DataFrame(rows_feat)
        batch['nama_pos_cat'] = batch['nama_pos_cat'].astype('category').cat.set_categories(pos_list)
        X_batch = batch[all_feats]
        preds = model.predict(X_batch)

        if clip_bounds is not None:
            clipped = []
            for p, pred in zip(batch['nama_pos'], preds):
                lo, hi = clip_bounds.get(p, (-np.inf, np.inf))
                clipped.append(min(max(pred, lo), hi))
            preds = np.array(clipped)

        batch['pred'] = preds
        for p, pred in zip(batch['nama_pos'], batch['pred']):
            series_by_pos[p].loc[ts] = pred

        results.append(batch[['nama_pos', 'datetime', 'pred']])

    return pd.concat(results, ignore_index=True)

out_dir = OUTPUT_DIR

full = pd.read_parquet(out_dir / 'full_grid.parquet')
exog = pd.read_parquet(out_dir / 'exog_features.parquet')
full = full.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)
exog = exog.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)
df = full.merge(exog, on=['nama_pos', 'datetime'], how='left')

df['hour'] = df['datetime'].dt.hour
df['month'] = df['datetime'].dt.month
doy = df['datetime'].dt.dayofyear
df['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
df['nama_pos_cat'] = df['nama_pos'].astype('category')

with open(out_dir / 'feature_cols.pkl', 'rb') as f:
    fc = pickle.load(f)
feature_cols, cat_features, all_feats = fc['feature_cols'], fc['cat_features'], fc['all_feats']
exog_cols = [c for c in exog.columns if c not in ('nama_pos', 'datetime')]

val_cutoff = pd.Timestamp(VAL_CUTOFF)

# ---- Train a model using ONLY data before val_cutoff (mirrors 03's tr split) ----
df_lag = add_lag_features(df)
train_only = df_lag[(df_lag['split'] == 'train') & (df_lag['datetime'] < val_cutoff)]
X_tr, y_tr = train_only[all_feats], train_only['tma_mdpl']
lgb_tr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_features)
params = dict(
    objective='regression', metric='rmse', learning_rate=0.03,
    num_leaves=63, min_data_in_leaf=30, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=1, verbose=-1,
)
print("Training model on pre-cutoff data only ...")
model = lgb.train(params, lgb_tr, num_boost_round=fc.get('best_iteration', 334))

# ---- Recursive forecast over the validation window itself ----
work = df.copy().sort_values(['nama_pos', 'datetime']).set_index(['nama_pos', 'datetime']).sort_index()
pos_list = work.index.get_level_values(0).unique()

val_mask = (df['split'] == 'train') & (df['datetime'] >= val_cutoff)
val_datetimes = sorted(df.loc[val_mask, 'datetime'].unique())
truth = df.loc[val_mask, ['nama_pos', 'datetime', 'tma_mdpl']].copy()

print(f"Recursive forecasting over {len(val_datetimes)} held-out validation timestamps "
      f"(this simulates the same process as the real test forecast)...")

# --- Version A: no clipping (mirrors current submission's method) ---
preds_noclip = recursive_forecast(model, work, exog_cols, all_feats, val_datetimes, pos_list, clip_bounds=None)

# --- Compute Tukey-fence clip bounds per pos from TRAIN data only ---
train_for_fence = df[(df['split'] == 'train') & (df['datetime'] < val_cutoff)]
clip_bounds = {}
for p, g in train_for_fence.groupby('nama_pos'):
    q1, q3 = g['tma_mdpl'].quantile([0.25, 0.75])
    iqr = q3 - q1
    clip_bounds[p] = (q1 - FENCE_K * iqr, q3 + FENCE_K * iqr)

# --- Version B: with clipping ---
preds_clip = recursive_forecast(model, work, exog_cols, all_feats, val_datetimes, pos_list, clip_bounds=clip_bounds)

# ---- Compare RMSE ----
def eval_rmse(preds, label):
    merged = truth.merge(preds, on=['nama_pos', 'datetime'], how='left')
    overall = np.sqrt(mean_squared_error(merged['tma_mdpl'], merged['pred']))
    per_pos = merged.groupby('nama_pos').apply(
        lambda g: np.sqrt(mean_squared_error(g['tma_mdpl'], g['pred']))
    ).sort_values(ascending=False)
    print(f"\n=== {label}: overall recursive RMSE = {overall:.4f} ===")
    print("Worst 8 pos:\n", per_pos.head(8))
    return overall, per_pos

overall_noclip, per_pos_noclip = eval_rmse(preds_noclip, "WITHOUT clipping")
overall_clip, per_pos_clip = eval_rmse(preds_clip, "WITH Tukey-fence clipping")

print(f"\n>>> Improvement from clipping: {overall_noclip:.4f} -> {overall_clip:.4f} "
      f"({'better' if overall_clip < overall_noclip else 'worse'})")

comp = pd.DataFrame({'no_clip': per_pos_noclip, 'with_clip': per_pos_clip})
comp['improved'] = comp['with_clip'] < comp['no_clip']
print("\nPer-pos comparison (sorted by no_clip RMSE desc):\n", comp.sort_values('no_clip', ascending=False).head(15))

with open(out_dir / 'clip_bounds.pkl', 'wb') as f:
    pickle.dump(clip_bounds, f)
print(f"\nSaved clip_bounds.pkl to {out_dir}")

Training model on pre-cutoff data only ...


Recursive forecasting over 369 held-out validation timestamps (this simulates the same process as the real test forecast)...



=== WITHOUT clipping: overall recursive RMSE = 1.9594 ===
Worst 8 pos:
 nama_pos
Lorog                6.851519
Karangnongko         3.119277
Wonogiri Dam         3.095481
Cepu                 2.853967
Jurug                2.673925
Sumberrejo           1.979343
Floodway Bridge C    1.975961
Napel                1.899361
dtype: float64

=== WITH Tukey-fence clipping: overall recursive RMSE = 1.4682 ===
Worst 8 pos:
 nama_pos
Karangnongko         3.119342
Wonogiri Dam         3.095481
Cepu                 2.854478
Jurug                2.673925
Floodway Bridge C    1.975961
Napel                1.899361
Karanggeneng         1.772548
Sekayu               1.755326
dtype: float64

>>> Improvement from clipping: 1.9594 -> 1.4682 (better)

Per-pos comparison (sorted by no_clip RMSE desc):
                            no_clip  with_clip  improved
nama_pos                                               
Lorog                     6.851519   0.226433      True
Karangnongko              3.119277   3.

## 7. Training final + forecast recursive + submission

Retrain model di SELURUH data train (tanpa holdout), lalu jalankan forecast
recursive step-by-step untuk seluruh periode test (726 timestamp x 30 pos),
dengan Tukey-fence clipping otomatis aktif untuk mencegah drift.


In [8]:
import argparse
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import lightgbm as lgb

pd.set_option('display.width', 160)

LAGS = [1, 2, 3, 4, 7, 28]


def add_lag_features(d):
    d = d.sort_values(['nama_pos', 'datetime']).copy()
    g = d.groupby('nama_pos')['tma_mdpl']
    for lag in LAGS:
        d[f'lag_{lag}'] = g.shift(lag)
    shifted = g.shift(1)
    d['roll_mean_4'] = shifted.groupby(d['nama_pos']).transform(lambda s: s.rolling(4, min_periods=1).mean())
    d['roll_mean_28'] = shifted.groupby(d['nama_pos']).transform(lambda s: s.rolling(28, min_periods=1).mean())
    d['roll_std_28'] = shifted.groupby(d['nama_pos']).transform(lambda s: s.rolling(28, min_periods=2).std())
    d['diff_1'] = d['lag_1'] - d['lag_2']

    if 'upstream_pos' in d.columns:
        up_cols = ['nama_pos', 'datetime', 'lag_1', 'lag_2', 'lag_3', 'diff_1', 'roll_mean_4']
        up_data = d[up_cols].rename(columns={
            'nama_pos': 'upstream_pos',
            'lag_1': 'up_lag_1', 'lag_2': 'up_lag_2', 'lag_3': 'up_lag_3',
            'diff_1': 'up_diff_1', 'roll_mean_4': 'up_roll_mean_4'
        })
        d = d.merge(up_data, on=['upstream_pos', 'datetime'], how='left')
    return d


def get_lag_val(series, ts, steps_back):
    idx = series.index
    pos_loc = idx.searchsorted(ts)
    target_loc = pos_loc - steps_back
    if target_loc < 0:
        return np.nan
    return series.iloc[target_loc]

data_dir = DATA_DIR
out_dir = OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

full = pd.read_parquet(out_dir / 'full_grid.parquet')
exog = pd.read_parquet(out_dir / 'exog_features.parquet')

full = full.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)
exog = exog.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

df = full.merge(exog, on=['nama_pos', 'datetime'], how='left')

df['hour'] = df['datetime'].dt.hour
df['month'] = df['datetime'].dt.month
doy = df['datetime'].dt.dayofyear
df['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
df['nama_pos_cat'] = df['nama_pos'].astype('category')

with open(out_dir / 'feature_cols.pkl', 'rb') as f:
    fc = pickle.load(f)
feature_cols, cat_features, all_feats = fc['feature_cols'], fc['cat_features'], fc['all_feats']
best_iteration = fc.get('best_iteration') or 380

# ---------- Step 1: retrain final model on ALL train rows (no holdout) ----------
df_lag = add_lag_features(df)
train_df = df_lag[df_lag['split'] == 'train'].copy()

X_all, y_all = train_df[all_feats], train_df['tma_mdpl']
lgb_all = lgb.Dataset(X_all, label=y_all, categorical_feature=cat_features)

params = dict(
    objective='regression', metric='rmse', learning_rate=0.03,
    num_leaves=63, min_data_in_leaf=30, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=1, verbose=-1,
)

print(f"Training final model for {best_iteration} rounds ...")
final_model = lgb.train(params, lgb_all, num_boost_round=best_iteration)
final_model.save_model(str(out_dir / 'lgb_model_final.txt'))
print("Final model trained on all train rows.")

# ---------- Step 2: recursive forecasting for test period ----------
work = df.copy()
work = work.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

test_mask = work['split'] == 'test'
test_datetimes = sorted(work.loc[test_mask, 'datetime'].unique())
print(f"Recursive forecasting over {len(test_datetimes)} timestamps x pos...")

work = work.set_index(['nama_pos', 'datetime']).sort_index()
pos_list = work.index.get_level_values(0).unique()

series_by_pos = {p: work.xs(p, level=0)['tma_mdpl'].sort_index() for p in pos_list}
static_row_cache = {p: work.xs(p, level=0) for p in pos_list}

# ---- Tukey-fence clip bounds per pos, computed from ALL train data ----
# Prevents runaway drift during long recursive forecasts, especially for
# near-constant stations (see 05_recursive_validation.py for why this matters).
clip_bounds = {}
for p, g in train_df.groupby('nama_pos'):
    q1, q3 = g['tma_mdpl'].quantile([0.25, 0.75])
    iqr = q3 - q1
    clip_bounds[p] = (q1 - FENCE_K * iqr, q3 + FENCE_K * iqr)
print(f"Computed clip bounds (fence_k={FENCE_K}) for {len(clip_bounds)} pos.")

results = []
for i, ts in enumerate(test_datetimes):
    rows_feat = []
    for p in pos_list:
        srow = static_row_cache[p]
        if ts not in srow.index:
            continue
        series = series_by_pos[p]
        lag_vals = {f'lag_{l}': get_lag_val(series, ts, l) for l in LAGS}

        idxloc = series.index.searchsorted(ts)
        past = series.iloc[max(0, idxloc - 28):idxloc]
        past4 = past.iloc[-4:] if len(past) >= 1 else past
        roll_mean_4 = past4.mean() if len(past4) > 0 else np.nan
        roll_mean_28 = past.mean() if len(past) > 0 else np.nan
        roll_std_28 = past.std() if len(past) > 1 else np.nan
        diff_1 = (lag_vals['lag_1'] - lag_vals['lag_2']
                  if pd.notna(lag_vals['lag_1']) and pd.notna(lag_vals['lag_2']) else np.nan)

        row = srow.loc[ts].copy()
        feat_row = {c: row[c] for c in exog.columns if c not in ('nama_pos', 'datetime')}
        feat_row.update(lag_vals)
        feat_row['roll_mean_4'] = roll_mean_4
        feat_row['roll_mean_28'] = roll_mean_28
        feat_row['roll_std_28'] = roll_std_28
        feat_row['diff_1'] = diff_1

        # ---- upstream features (safe: only uses upstream pos's t-1/t-2/t-3,
        # which are already finalized before we compute pos p at time ts) ----
        up_pos = row.get('upstream_pos', None)
        if pd.notna(up_pos) and up_pos in series_by_pos:
            up_series = series_by_pos[up_pos]
            up_lag1 = get_lag_val(up_series, ts, 1)
            up_lag2 = get_lag_val(up_series, ts, 2)
            up_lag3 = get_lag_val(up_series, ts, 3)
            up_idxloc = up_series.index.searchsorted(ts)
            up_past4 = up_series.iloc[max(0, up_idxloc - 1 - 4):up_idxloc - 1] if up_idxloc >= 1 else pd.Series(dtype=float)
            feat_row['up_lag_1'] = up_lag1
            feat_row['up_lag_2'] = up_lag2
            feat_row['up_lag_3'] = up_lag3
            feat_row['up_diff_1'] = (up_lag1 - up_lag2) if pd.notna(up_lag1) and pd.notna(up_lag2) else np.nan
            feat_row['up_roll_mean_4'] = up_past4.mean() if len(up_past4) > 0 else np.nan
        else:
            feat_row['up_lag_1'] = np.nan
            feat_row['up_lag_2'] = np.nan
            feat_row['up_lag_3'] = np.nan
            feat_row['up_diff_1'] = np.nan
            feat_row['up_roll_mean_4'] = np.nan
        feat_row['dist_to_upstream_km'] = row.get('dist_to_upstream_km', np.nan)

        feat_row['hour'] = ts.hour
        feat_row['month'] = ts.month
        doy = ts.dayofyear
        feat_row['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
        feat_row['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
        feat_row['latitude'] = row['latitude']
        feat_row['longitude'] = row['longitude']
        feat_row['nama_pos_cat'] = p
        feat_row['nama_pos'] = p
        feat_row['datetime'] = ts
        rows_feat.append(feat_row)

    batch = pd.DataFrame(rows_feat)
    batch['nama_pos_cat'] = batch['nama_pos_cat'].astype('category').cat.set_categories(pos_list)
    X_batch = batch[all_feats]
    preds = final_model.predict(X_batch)

    clipped_preds = []
    for p, pred in zip(batch['nama_pos'], preds):
        lo, hi = clip_bounds.get(p, (-np.inf, np.inf))
        clipped_preds.append(min(max(pred, lo), hi))
    batch['pred'] = clipped_preds

    for p, pred in zip(batch['nama_pos'], batch['pred']):
        series_by_pos[p].loc[ts] = pred

    results.append(batch[['nama_pos', 'datetime', 'pred']])

    if (i + 1) % 50 == 0 or i == len(test_datetimes) - 1:
        print(f"  {i+1}/{len(test_datetimes)} timestamps done")

test_preds = pd.concat(results, ignore_index=True)
test_preds.to_parquet(out_dir / 'test_predictions.parquet', index=False)
print("Saved test_predictions.parquet, shape:", test_preds.shape)

# ---------- Step 3: build submission.csv ----------
test_raw = pd.read_csv(data_dir / 'test.csv')
test = pd.DataFrame()
test['id'] = test_raw['id']
test['datetime'] = pd.to_datetime(test_raw['id'].str.split(' - ', n=1).str[0])
test['nama_pos'] = test_raw['id'].str.split(' - ', n=1).str[1]

sub = test.merge(test_preds, on=['nama_pos', 'datetime'], how='left')
missing = sub['pred'].isna().sum()
if missing:
    print(f"WARNING: {missing} missing predictions in submission!")
sub['tma_mdpl'] = sub['pred'].round(4)
submission = sub[['id', 'tma_mdpl']]

sub_path = out_dir / 'submission.csv'
submission.to_csv(sub_path, index=False)
print(f"Saved {sub_path}")
print(submission.head())

Training final model for 328 rounds ...


Final model trained on all train rows.


Recursive forecasting over 726 timestamps x pos...
Computed clip bounds (fence_k=3.0) for 30 pos.


  50/726 timestamps done


  100/726 timestamps done


...

  550/726 timestamps done


  600/726 timestamps done


  650/726 timestamps done


  700/726 timestamps done


  726/726 timestamps done
Saved test_predictions.parquet, shape: (21780, 3)
Saved output/submission.csv
                                             id  tma_mdpl
0  2025-09-19 06:00:00 - Arjowinangun - Pacitan    1.2443
1  2025-09-19 12:00:00 - Arjowinangun - Pacitan    0.9547
2  2025-09-19 18:00:00 - Arjowinangun - Pacitan    1.0236
3  2025-09-20 06:00:00 - Arjowinangun - Pacitan    1.3396
4  2025-09-20 12:00:00 - Arjowinangun - Pacitan    1.1466


## 8. Verifikasi submission.csv

Cek format, kecocokan `id` dengan `sample_submission.csv`, dan kewajaran
nilai per pos sebelum benar-benar disubmit ke Kaggle.


In [9]:
sub = pd.read_csv(OUTPUT_DIR / 'submission.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
train_check = pd.read_csv(DATA_DIR / 'train.csv')

print("Shape:", sub.shape, "(harus: (21780, 2))")
print("NaN:", sub['tma_mdpl'].isna().sum(), "| Duplikat id:", sub['id'].duplicated().sum())
print("Urutan id cocok dg sample_submission?", (sub['id'].values == sample['id'].values).all())

sub['nama_pos'] = sub['id'].str.split(' - ', n=1).str[1]
train_range = train_check.groupby('nama_pos')['tma_mdpl'].agg(['min', 'max'])
sub_range = sub.groupby('nama_pos')['tma_mdpl'].agg(['min', 'max'])
compare = train_range.join(sub_range, lsuffix='_train', rsuffix='_sub')
compare['out_of_range'] = (compare['min_sub'] < compare['min_train'] - 0.5) | \
                          (compare['max_sub'] > compare['max_train'] * 1.3 + 0.5)
print("\nPos di luar batas wajar historis:", compare['out_of_range'].sum())
print(compare[compare['out_of_range']])

Shape: (21780, 2) (harus: (21780, 2))
NaN: 0 | Duplikat id: 0
Urutan id cocok dg sample_submission? True

Pos di luar batas wajar historis: 0
Empty DataFrame
Columns: [min_train, max_train, min_sub, max_sub, out_of_range]
Index: []
